In [ ]:
def optimize(
    camera: CodedMaskCamera,
    sky: npt.NDArray,
    arg_sky: tuple[int, int],
    vignetting: bool = True,
    psfy: bool = True,
    verbose: bool = False,
) -> tuple[float, float, float]:
    """
    Performs the optimization to fit a point source model to sky image data.
    """
    px_dim_x, px_dim_y = (
        camera.specs.mask_deltax / camera.upscale_f.x,
        camera.specs.mask_deltay / camera.upscale_f.y,
    )
    camera_coding_power = 0.85

    model_shift_flux = _ModelShiftFluence(camera, arg_sky, vignetting, psfy)
    sx_start, sy_start = pos2shift(camera, *arg_sky)
    sky_peak = sky[*arg_sky]
    fluence_start = (
        sky_peak / camera_coding_power if psfy else sky_peak
    )
    sky_ydata = process_skyimg(camera, sky, arg_sky)
    sky_ydata_sigma = np.ones_like(sky_ydata)
    _posvals = (sky_ydata > 0)
    sky_ydata_sigma[_posvals] = 1.0 / np.sqrt(sky_ydata[_posvals])
    
    # - the shifts are allowed to fluctuate in a small pixel box since
    #   the extracted position is close enough to the true source pos
    #   Also, since multiple sources may be superimposed or close, the
    #   optimisation procedure may introduce biases in the source fit
    # - the fluence cannot be smaller than the one observed at the peak,
    #   and we insert a lower value just for precaution (if simulating
    #   for example an infinite detector spatial resolution)
    dx, dy = camera.upscale_f.x, camera.upscale_f.y
    results, cov, info, msg, _ = curve_fit(
        model_shift_flux,
        xdata=np.arange(len(sky_ydata)),
        ydata=sky_ydata,
        p0=[sx_start, sy_start, fluence_start],
        sigma=sky_ydata_sigma,
        bounds=[
            (
                max(sx_start - dx * px_dim_x, camera.bins_sky.x[0]),
                max(sy_start - dy * px_dim_y, camera.bins_sky.y[0]),
                0.95 * sky_peak,
            ),
            (
                min(sx_start + dx * px_dim_x, camera.bins_sky.x[-1]),
                min(sy_start + dy * px_dim_y, camera.bins_sky.y[-1]),
                1.25 * sky_peak,
            ),
        ],
        full_output=True,
    )
    # store the final optimized positions and fluence
    sx, sy, fluence = map(float, results)

    if verbose:
        print(
            f'\n'
            f'## Optimisation Results:\n'
            f'  - fluence START: {fluence_start}\n'
            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'

            f'  - fluence OPTIM.: {fluence}\n'
            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'

            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
            f'  - shift_x GAIN %: {np.sign(sx_start) * (sx - sx_start) * 100 / sx_start:.3f}\n'
            f'  - shift_y GAIN %: {np.sign(sy_start) * (sy - sy_start) * 100 / sy_start:.3f}\n'
        )

    return sx, sy, fluence